### cell A：paths + load data + constants

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

def find_project_root(start: Path) -> Path:
    """Walk upward until we find 'outcomes' folder (repo marker)."""
    cur = start.resolve()
    for _ in range(12):
        if (cur / "outcomes").exists():
            return cur
        cur = cur.parent
    return start.resolve()

project_root = find_project_root(Path.cwd())

in_fp = project_root / "outcomes" / "s1" / "cleaned_dataset_row2_imputed.csv"
out_dir = project_root / "outcomes" / "s4"   # <- force s4
out_dir.mkdir(parents=True, exist_ok=True)

print("project_root:", project_root)
print("input:", in_fp)
print("output dir:", out_dir)
assert in_fp.exists(), f"Input file not found: {in_fp}"

df = pd.read_csv(in_fp)

TARGET = "Readmission"
RANDOM_STATE = 103  # keep stable baseline for CV object
BEST_SEED = 103
BEST_TEST_SIZE = 0.2

FIXED_RF_PARAMS = dict(
    bootstrap=True,
    class_weight="balanced_subsample",
    max_depth=None,
    max_features="sqrt",
    min_impurity_decrease=1e-4,
    min_samples_leaf=20,
    min_samples_split=2,
    n_estimators=1000,
    n_jobs=-1,
)

# Candidate features: the current pool of 19 significant variables, in the original order.
CANDIDATE_FEATURES = [
    "Albumin", "Alcohol Use", "Hypertension", "Immunocompromised", "CVD",
    "Num Dis Vessels", "Prior MI", "Hematocrit", "PAD", "Sleep Apnea",
    "Creatinine", "Gender", "Pneumonia", "tobacco_use", "race",
    "Diabetes", "MELD Score", "CLD", "Status"
]

need = [TARGET] + CANDIDATE_FEATURES
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in df: {missing}")

X_all = df[CANDIDATE_FEATURES].copy()
y_all = df[TARGET].astype(int).copy()

print("Total n:", len(df))
print("Event rate:", float(y_all.mean()))
print("Candidate features:", len(CANDIDATE_FEATURES))


### Cell B: Split the data (use training data only for forward-selection CV)

In [ ]:
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_all
)

print("Train n:", len(X_train_all), "Test n:", len(X_test_all))
print("Train event rate:", float(y_train.mean()), "Test event rate:", float(y_test.mean()))


### Cell C: Compute CV AUROC with the selected Random Forest hyperparameters

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Use the selected RF hyperparameters as the base model for a fair forward-selection comparison.
base_rf = RandomForestClassifier(
    random_state=RANDOM_STATE,   # Fix the seed to make comparisons reproducible.
    **FIXED_RF_PARAMS
)

def cv_auc_for_features(feats):
    Xtr = X_train_all[feats]
    scores = cross_val_score(
        base_rf,
        Xtr, y_train,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1
    )
    return float(np.mean(scores)), float(np.std(scores, ddof=1)), scores


### Cell D: Forward selection (generate the selection history and save a CSV)

In [ ]:
t0 = time.time()

remaining = list(CANDIDATE_FEATURES)
selected = []
history = []

target_k = len(CANDIDATE_FEATURES)  # Evaluate all candidate features, then select the best k.

while len(selected) < target_k:
    best_feat = None
    best_score = -np.inf
    best_sd = None

    for f in remaining:
        trial = selected + [f]
        m, sd, _ = cv_auc_for_features(trial)
        if m > best_score:
            best_score = m
            best_sd = sd
            best_feat = f

    selected.append(best_feat)
    remaining.remove(best_feat)
    history.append((len(selected), best_score, best_sd, best_feat))

    print(f"step {len(selected):02d}: add {best_feat} -> CV AUROC = {best_score:.4f} (sd={best_sd:.4f})")

hist_df = pd.DataFrame(history, columns=["k", "cv_auc_at_step", "cv_auc_sd_at_step", "added_feature"])

hist_path = out_dir / "rf_forward_selection_history.csv"
hist_df.to_csv(hist_path, index=False)

print("\nDone. time(s):", round(time.time() - t0, 1))
print("Saved:", hist_path)

print("\nSelected order:")
print(selected)


### Cell E: Recalculate CV AUROC for each k (curve and best-k selection)

In [ ]:
rows = []
best_k = None
best_cv = -np.inf
best_feats = None

for k in range(1, len(selected) + 1):
    feats_k = selected[:k]
    m, sd, _ = cv_auc_for_features(feats_k)
    rows.append({
        "k": k,
        "cv_auc_mean": m,
        "cv_auc_sd": sd,
        "features": "|".join(feats_k)
    })
    print(f"k={k:02d} CV AUROC={m:.4f}±{sd:.4f}")

    if m > best_cv:
        best_cv = m
        best_k = k
        best_feats = feats_k

k_df = pd.DataFrame(rows)
k_path = out_dir / "rf_forward_selection_k_sweep.csv"
k_df.to_csv(k_path, index=False)

print("\n=== Best k by CV AUROC ===")
print("best_k:", best_k)
print("best_cv_auc:", round(best_cv, 4))
print("best_features:", best_feats)

print("Saved:", k_path)


### Cell F: The RFECV curve in Cell 6 can now be generated

In [ ]:
import matplotlib.pyplot as plt

def plot_rfecv_curve(xs, mean, std, best_n, best_score, title, out_path, ylab):
    plt.figure(figsize=(8, 5.5))
    plt.plot(xs, mean, "o-", label=f"Mean {ylab}")
    if std is not None:
        plt.fill_between(xs, mean - std, mean + std, alpha=0.2, label="± 1 Std")
    plt.axvline(best_n, color="k", linestyle="--",
                label=f"n_features = {best_n}\nscore = {best_score:.3f}")
    plt.title(title)
    plt.xlabel("Number of Features Selected")
    plt.ylabel(ylab)
    plt.legend(loc="lower right")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_path, dpi=250)
    plt.close()

k_df = pd.read_csv(out_dir / "rf_forward_selection_k_sweep.csv")
xs = k_df["k"].values
mean = k_df["cv_auc_mean"].values
std = k_df["cv_auc_sd"].values

best_idx = int(np.argmax(mean))
best_n = int(xs[best_idx])
best_score = float(mean[best_idx])

out_path = out_dir / "rf_feature_selection_curve_auc.png"
plot_rfecv_curve(
    xs=xs,
    mean=mean,
    std=std,
    best_n=best_n,
    best_score=best_score,
    title="Forward selection (train CV AUROC) vs #features",
    out_path=out_path,
    ylab="AUROC"
)

print("Saved:", out_path)
print("Best n_features =", best_n, "best CV AUROC =", round(best_score, 4))


### Cell 7: Use the selected split and RF model for SHAP, dependence, interaction, and force plots

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# -----------------------
# 0) Load best_n and best_features from saved CSVs
# -----------------------
k_sweep_fp = out_dir / "rf_forward_selection_k_sweep.csv"
hist_fp    = out_dir / "rf_forward_selection_history.csv"

assert k_sweep_fp.exists(), f"Missing: {k_sweep_fp}"
assert hist_fp.exists(),    f"Missing: {hist_fp}"

k_df = pd.read_csv(k_sweep_fp)
hist = pd.read_csv(hist_fp)

# best_n from k-sweep
xs = k_df["k"].values
mean = k_df["cv_auc_mean"].values
best_idx = int(np.argmax(mean))
best_n = int(xs[best_idx])
best_score = float(mean[best_idx])

# features from history (selection order)
# history expected columns: ["k","cv_auc_at_step","added_feature"]
if "added_feature" not in hist.columns:
    raise ValueError(f"'added_feature' not found in {hist_fp}. Columns={list(hist.columns)}")

BEST_FEATURES = hist["added_feature"].tolist()[:best_n]

print("Best n_features =", best_n, "best train-CV AUROC =", round(best_score, 4))
print("BEST_FEATURES (from history top best_n):", BEST_FEATURES)

# sanity
missing_feats = [c for c in BEST_FEATURES if c not in df.columns]
if missing_feats:
    raise ValueError(f"Some selected features not in df.columns: {missing_feats}")

# -----------------------
# 1) Split using your chosen seed + test_size, then fit best RF
# -----------------------
X_all = df[BEST_FEATURES].copy()
y_all = df[TARGET].astype(int).copy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all,
    test_size=BEST_TEST_SIZE,
    random_state=BEST_SEED,
    stratify=y_all
)

best_rf = RandomForestClassifier(
    random_state=BEST_SEED,
    **FIXED_RF_PARAMS
)
best_rf.fit(X_tr, y_tr)

print("Train n:", len(X_tr), "Test n:", len(X_te))
print("Train event rate:", float(y_tr.mean()), "Test event rate:", float(y_te.mean()))

# -----------------------
# 2) Pretty name mapping (works for your title-case column names too)
# -----------------------
def normalize_name(x: str) -> str:
    x = str(x).strip().lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

PRETTY_MAP_RAW = {
    "albumin":"Albumin",
    "bilirubin":"Bilirubin",
    "creatinine":"Creatinine",
    "diabetes":"Diabetes",
    "endocarditis":"Endocarditis",
    "gender":"Gender",
    "hematocrit":"Hematocrit",
    "hemoglobin":"Hemoglobin",
    "home_oxygen":"Home Oxygen",
    "hypertension":"Hypertension",
    "meld_score":"MELD Score",
    "mediastinal_radiation":"Mediastinal Radiation",
    "platelets":"Platelets",
    "pneumonia":"Pneumonia",
    "race":"Race",
    "sleep_apnea":"Sleep Apnea",
    "syncope":"Syncope",
    "a1c_level":"Hemoglobin A1c",
    "alcohol_use":"Alcohol use",
    "bmi":"Body mass index",
    "cld":"Chronic lung disease",
    "cvd":"Cerebrovascular disease",
    "hemo_data_ef":"Ejection Fraction",
    "illicit_drug_use":"Illicit drug use",
    "immunocompromise":"Immunocompromised",
    "liver_disease":"Liver disease",
    "num_dis_vessels":"Number of Diseased Vessels",
    "pad":"Peripheral artery disease",
    "patient_age":"Age",
    "prev_cardiac_intervent":"Previous cardiac intervention",
    "previous_pci":"Previous cardiac intervention",
    "prior_mi":"Prior myocardial Infarction",
    "status":"Admission status",
    "tobacco_use":"Tobacco use",
    "wbc_count":"White blood cell",
}

PRETTY_MAP = {normalize_name(k): v for k, v in PRETTY_MAP_RAW.items()}

def pretty_name(col: str, max_len: int = 26) -> str:
    key = normalize_name(col)
    text = PRETTY_MAP.get(key, str(col))
    if len(text) > max_len:
        text = text[:max_len-1] + "…"
    return text

# -----------------------
# 3) SHAP explainer + fix to class=1 2D
# -----------------------
explainer = shap.TreeExplainer(
    best_rf,
    data=X_tr,
    model_output="probability",
    feature_perturbation="interventional"
)

sv_tr = explainer(X_tr, check_additivity=False)
sv_te = explainer(X_te, check_additivity=False)

def fix_shap_values(sv, X):
    vals = sv.values
    if isinstance(vals, list):
        vals2d = vals[1]
    elif getattr(vals, "ndim", 2) == 3 and vals.shape[-1] == 2:
        vals2d = vals[..., 1]
    else:
        vals2d = vals

    base = sv.base_values
    if np.ndim(base) == 2 and base.shape[1] == 2:
        base = base[:, 1]

    return shap.Explanation(
        values=vals2d,
        base_values=base,
        data=X.to_numpy(),
        feature_names=list(X.columns)
    )

sv_tr_fixed = fix_shap_values(sv_tr, X_tr)
sv_te_fixed = fix_shap_values(sv_te, X_te)

# -----------------------
# 4) Beeswarm (train)
# -----------------------
sv_show = shap.Explanation(
    values=sv_tr_fixed.values,
    base_values=sv_tr_fixed.base_values,
    data=sv_tr_fixed.data,
    feature_names=[pretty_name(c) for c in X_tr.columns]
)

plt.figure(figsize=(8, 5.5))
shap.plots.beeswarm(sv_show, show=False, max_display=min(20, X_tr.shape[1]))
plt.gcf().subplots_adjust(left=0.35)
plt.tight_layout()

beeswarm_path = out_dir / "rf_shap_beeswarm.png"
plt.savefig(beeswarm_path, dpi=240)
plt.close()
print("Saved:", beeswarm_path)

# -----------------------
# 5) Stacked bar by truth (train)
# -----------------------
vals_abs = np.abs(np.asarray(sv_tr_fixed.values))
y_arr = np.asarray(y_tr).astype(int)

if (y_arr == 0).sum() > 0 and (y_arr == 1).sum() > 0:
    m0 = vals_abs[y_arr == 0].mean(axis=0)
    m1 = vals_abs[y_arr == 1].mean(axis=0)
    total = m0 + m1
    order = np.argsort(total)[::-1][:min(15, X_tr.shape[1])]

    feats = [pretty_name(c) for c in X_tr.columns[order]]
    b0 = m0[order]; b1 = m1[order]
    yy = np.arange(len(order))

    plt.figure(figsize=(8, 5.5))
    plt.barh(yy, b0, label="Readmission = No")
    plt.barh(yy, b1, left=b0, label="Readmission = Yes")
    plt.yticks(yy, feats)
    plt.xlabel("|SHAP| mean (per truth group)")
    plt.ylabel("Feature")
    plt.legend(loc="lower right")
    plt.gcf().subplots_adjust(left=0.38)
    plt.gca().invert_yaxis()
    plt.tight_layout()

    stack_path = out_dir / "rf_shap_stacked_bar_classes.png"
    plt.savefig(stack_path, dpi=260)
    plt.close()
    print("Saved:", stack_path)
else:
    print("Skip stacked bar: one of the truth groups is empty.")

# -----------------------
# 6) Top-6 by mean |SHAP| (train) -> dependence + interaction
# -----------------------
top6_idx = np.argsort(np.abs(sv_tr_fixed.values).mean(axis=0))[::-1][:6]
top6_cols = list(X_tr.columns[top6_idx])
print("Top 6 by mean |SHAP|:", top6_cols)

# dependence
dep_dir = out_dir / "rf_shap_dependence_top6"
dep_dir.mkdir(parents=True, exist_ok=True)

for col in top6_cols:
    plt.figure(figsize=(7.5, 5.2))
    shap.plots.scatter(sv_tr_fixed[:, col], color=sv_tr_fixed, show=False)
    plt.title(f"Dependence: {pretty_name(col)}")
    plt.tight_layout()
    fp = dep_dir / f"dependence_{normalize_name(col)}.png"
    plt.savefig(fp, dpi=250)
    plt.close()
print("Saved dependence plots to:", dep_dir)




In [ ]:
# =========================
# PATCH for cell 7:
# 1) Make interaction plots actually save (common SHAP pitfalls handled)
# 2) Replace your force-plot block with the "example style" (top_pad, waterfall fallback)
# =========================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import shap
import warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# A) Interaction plots (robust version)
# Why yours often fails:
# - shap_interaction_values can be huge / slow / error depending on SHAP version
# - for RF, sometimes returns unexpected shapes
# - dependence_plot can error if it can't find feature name indices properly
# Fix: compute interaction values once; enforce ndarray shape (n, p, p); use safe indexing;
#      and add a fallback that still produces a plot even if interaction fails.
# ------------------------------------------------------------
# =========================
# FIX: Interaction plots MUST use model_output="raw"
# Replace your interaction block with this one
# =========================

# =========================
# PATCH v2: interaction plots WITHOUT background dataset
# - avoids leaf coverage error
# - handles inter_vals shape (n,p,p,2)
# =========================

int_dir = out_dir / "rf_shap_interaction_top6"
int_dir.mkdir(parents=True, exist_ok=True)

try:
    # key fix: DO NOT pass data=...
    inter_explainer = shap.TreeExplainer(
        best_rf,
        model_output="raw",
        feature_perturbation="tree_path_dependent"
    )

    inter_vals = inter_explainer.shap_interaction_values(X_tr)

    # binary -> sometimes list; sometimes ndarray with last dim=2
    if isinstance(inter_vals, list):
        inter_vals = inter_vals[1]  # class=1
    inter_vals = np.asarray(inter_vals)

    # handle shape (n,p,p,2) -> take class=1
    if inter_vals.ndim == 4 and inter_vals.shape[-1] == 2:
        inter_vals = inter_vals[..., 1]

    if inter_vals.ndim != 3 or inter_vals.shape[1] != inter_vals.shape[2]:
        raise ValueError(f"Unexpected inter_vals shape: {inter_vals.shape}")

    # also need raw SHAP 2D for dependence_plot
    sv_raw = inter_explainer.shap_values(X_tr)
    if isinstance(sv_raw, list):
        sv_raw = sv_raw[1]
    sv_raw = np.asarray(sv_raw)

    # handle shape (n,p,2) -> class=1
    if sv_raw.ndim == 3 and sv_raw.shape[-1] == 2:
        sv_raw = sv_raw[..., 1]

    if sv_raw.ndim != 2:
        raise ValueError(f"Unexpected sv_raw shape: {sv_raw.shape}")

    # draw interactions: strongest partner per top feature
    for col in top6_cols:
        j = X_tr.columns.get_loc(col)
        m = np.abs(inter_vals[:, j, :]).mean(axis=0)
        m[j] = -np.inf
        partner = X_tr.columns[int(np.argmax(m))]

        plt.figure(figsize=(7.5, 5.2))
        shap.dependence_plot(
            ind=col,
            shap_values=sv_raw,      # raw shap values
            features=X_tr,
            interaction_index=partner,
            show=False
        )
        plt.title(f"Interaction: {pretty_name(col)} x {pretty_name(partner)}")
        plt.tight_layout()
        fp = int_dir / f"interaction_{normalize_name(col)}_x_{normalize_name(partner)}.png"
        plt.savefig(fp, dpi=250)
        plt.close()

    print("Saved interaction plots to:", int_dir)

except Exception as e:
    print("Interaction plots failed again:", repr(e))
    print("Fallback: saving dependence-only plots (no interaction coloring).")

    for col in top6_cols:
        plt.figure(figsize=(7.5, 5.2))
        shap.plots.scatter(sv_tr_fixed[:, col], color=sv_tr_fixed, show=False)
        plt.title(f"Dependence (fallback): {pretty_name(col)}")
        plt.tight_layout()
        fp = int_dir / f"interaction_fallback_dependence_{normalize_name(col)}.png"
        plt.savefig(fp, dpi=250)
        plt.close()

    print("Saved fallback plots to:", int_dir)





# ------------------------------------------------------------
# B) Force plots (replace your old section with the "example style")
# This matches your example: top_pad + waterfall fallback.
# ------------------------------------------------------------

force_dir = out_dir / "rf_forceplots_test_top_bottom3"
force_dir.mkdir(parents=True, exist_ok=True)

def plot_force_for_top_and_bottom_k(
    model,
    Xte: pd.DataFrame,
    out_dir: Path,
    k: int = 3,
    prefix_top: str = "force_toprisk",
    prefix_low: str = "force_lowrisk",
    width: float = 12.0,
    height: float = 2.8,
    top_pad: float = 1.05,
    save_dpi: int = 260,
):
    proba = model.predict_proba(Xte)[:, 1]
    n = len(proba)
    k_eff = min(k, n)

    top_idx = np.argsort(proba)[::-1][:k_eff]
    low_idx = np.argsort(proba)[:k_eff]

    explainer = shap.TreeExplainer(
        model,
        data=Xte,
        model_output="probability",
        feature_perturbation="interventional"
    )
    sv = explainer(Xte, check_additivity=False)

    # force plot needs a single-class 2D Explanation
    vals = sv.values
    if isinstance(vals, list):
        vals2d = vals[1]
    elif getattr(vals, "ndim", 2) == 3 and vals.shape[-1] == 2:
        vals2d = vals[..., 1]
    else:
        vals2d = vals

    base = sv.base_values
    if np.ndim(base) == 2 and base.shape[1] == 2:
        base = base[:, 1]

    sv_fixed = shap.Explanation(
        values=np.asarray(vals2d),
        base_values=np.asarray(base),
        data=Xte.to_numpy(),
        feature_names=list(Xte.columns)
    )

    def _save(indices, prefix):
        for i, ridx in enumerate(indices, start=1):
            try:
                plt.figure(figsize=(width, height))
                shap.plots.force(sv_fixed[ridx], matplotlib=True, show=False)
                plt.gcf().tight_layout(pad=0.6)
                plt.gcf().subplots_adjust(top=top_pad)
                fp = out_dir / f"{prefix}_{i:02d}.png"
                plt.savefig(fp, dpi=save_dpi, bbox_inches="tight", pad_inches=0.15)
                plt.close()
            except Exception as e:
                # fallback waterfall
                try:
                    plt.figure(figsize=(width, height + 1.0))
                    shap.plots.waterfall(sv_fixed[ridx], show=False, max_display=20)
                    plt.gcf().tight_layout(pad=0.8)
                    plt.gcf().subplots_adjust(top=min(top_pad + 0.05, 0.98))
                    fp = out_dir / f"{prefix}_{i:02d}_waterfall.png"
                    plt.savefig(fp, dpi=save_dpi, bbox_inches="tight", pad_inches=0.15)
                    plt.close()
                    print(f"[{prefix}] force -> waterfall fallback for #{i}: {e}")
                except Exception as e2:
                    print(f"[{prefix}] Skip plot #{i}: {e2}")

    _save(top_idx, prefix_top)
    _save(low_idx, prefix_low)

plot_force_for_top_and_bottom_k(
    best_rf,
    X_te,
    force_dir,
    k=3,
    prefix_top="force_toprisk",
    prefix_low="force_lowrisk",
)

print("Saved force plots to:", force_dir)


### Cell 8: Automatically generate Table 1 (p ≤ 0.05) as DOCX and CSV

In [ ]:
# =======================
# cell 8 (FINAL)
# 1) Table 1 uses ALL_CANDIDATES (NOT only BEST_FEATURES)
# 2) Continuous vars shown as mean ± sd
# 3) P value formatted to 3 decimals; if <0.01 show "<0.01"
# 4) Binary vars: show ONLY one row (the variable row) with "=1" counts (no extra indented row)
# =======================

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import chi2_contingency, fisher_exact, ttest_ind

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.shared import OxmlElement, qn

ALL_CANDIDATES = [
    "Albumin", "Alcohol Use", "Hypertension", "Immunocompromised", "CVD",
    "Num Dis Vessels", "Prior MI", "Hematocrit", "PAD", "Sleep Apnea",
    "Creatinine", "Gender", "Pneumonia", "tobacco_use", "race",
    "Diabetes", "MELD Score", "CLD", "Status"
]

# -----------------------
# 0) Sanity: required objects
# -----------------------
assert "df" in globals(), "df not found. Run the data-loading cell first."
assert "TARGET" in globals(), "TARGET not found."
assert "out_dir" in globals(), "out_dir not found."

# -----------------------
# 1) DOCX helpers
# -----------------------
def set_cell_margins(cell, top=80, start=80, bottom=80, end=80):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcMar = OxmlElement("w:tcMar")
    for m, v in (("top", top), ("start", start), ("bottom", bottom), ("end", end)):
        node = OxmlElement(f"w:{m}")
        node.set(qn("w:w"), str(v))
        node.set(qn("w:type"), "dxa")
        tcMar.append(node)
    tcPr.append(tcMar)

def build_table1_docx(
    df_: pd.DataFrame,
    out_path: Path,
    title: str = "Table 1. Baseline characteristics by readmission status (all variables; continuous = mean ± sd)",
):
    doc = Document()
    p = doc.add_paragraph(title)
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT

    table = doc.add_table(rows=1, cols=df_.shape[1])
    table.style = "Table Grid"

    # header
    hdr_cells = table.rows[0].cells
    for j, col in enumerate(df_.columns):
        hdr_cells[j].text = str(col)
        set_cell_margins(hdr_cells[j])

    # body
    for i in range(df_.shape[0]):
        row_cells = table.add_row().cells
        for j, col in enumerate(df_.columns):
            val = "" if pd.isna(df_.iloc[i, j]) else str(df_.iloc[i, j])
            row_cells[j].text = val
            set_cell_margins(row_cells[j])
            for par in row_cells[j].paragraphs:
                par.alignment = WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.RIGHT

        # keep indentation if any (we won't add indented rows for binary anymore,
        # but keep this in case you later want multi-level categorical rows)
        v = str(df_.iloc[i, 0])
        if v.startswith("  "):
            row_cells[0].text = v.strip()
            for par in row_cells[0].paragraphs:
                par.paragraph_format.left_indent = Inches(0.2)

    style = doc.styles["Normal"]
    style.font.name = "Times New Roman"
    style.font.size = Pt(10)

    doc.save(out_path)

# -----------------------
# 2) Table 1 builder
# -----------------------
use_cols = [TARGET] + list(ALL_CANDIDATES)
missing = [c for c in use_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in df: {missing}")

dat = df[use_cols].copy()
dat[TARGET] = dat[TARGET].astype(int)

g0 = dat[dat[TARGET] == 0]
g1 = dat[dat[TARGET] == 1]
n0, n1 = len(g0), len(g1)
if n0 == 0 or n1 == 0:
    raise ValueError("One group is empty; cannot build Table 1.")

def is_numeric_series(s: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(s)

def is_binary_series(s: pd.Series) -> bool:
    x = pd.to_numeric(s, errors="coerce").dropna().unique()
    x = np.unique(x)
    return len(x) <= 2 and set(x).issubset({0, 1})

def fmt_mean_sd(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return ""
    return f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

def fmt_n_pct(k, n):
    if n == 0:
        return ""
    return f"{int(k)} ({100.0*float(k)/float(n):.1f}%)"

def pvalue_numeric_ttest(x0, x1):
    x0 = pd.to_numeric(x0, errors="coerce").dropna()
    x1 = pd.to_numeric(x1, errors="coerce").dropna()
    if len(x0) < 2 or len(x1) < 2:
        return np.nan
    try:
        return float(ttest_ind(x0, x1, equal_var=False).pvalue)  # Welch
    except Exception:
        return np.nan

def pvalue_categorical(tab):
    try:
        if tab.shape == (2, 2) and (tab.values < 5).any():
            return float(fisher_exact(tab.values)[1])
        return float(chi2_contingency(tab.values)[1])
    except Exception:
        return np.nan

def fmt_p(p):
    if pd.isna(p):
        return ""
    p = float(p)
    if p < 0.01:
        return "<0.01"
    return f"{p:.3f}"

rows = []

for var in ALL_CANDIDATES:
    s = dat[var]

    # -----------------
    # Binary numeric -> ONE line only (show "=1" counts on the same row)
    # -----------------
    if is_numeric_series(s) and is_binary_series(s):
        c0 = pd.crosstab(dat[TARGET], dat[var], dropna=False)
        # ensure columns 0/1 exist
        for lv in [0, 1]:
            if lv not in c0.columns:
                c0[lv] = 0
        c0 = c0[[0, 1]]

        pv = pvalue_categorical(c0)

        k0_1 = int(c0.loc[0, 1]) if 0 in c0.index else 0
        k1_1 = int(c0.loc[1, 1]) if 1 in c0.index else 0

        # variable name includes "=1"
        rows.append([
            f"{var} = 1",
            fmt_n_pct(k0_1, n0),
            fmt_n_pct(k1_1, n1),
            fmt_p(pv)
        ])
        continue

    # -----------------
    # Continuous numeric -> mean ± sd
    # -----------------
    if is_numeric_series(s):
        pv = pvalue_numeric_ttest(g0[var], g1[var])
        rows.append([var, fmt_mean_sd(g0[var]), fmt_mean_sd(g1[var]), fmt_p(pv)])
        continue

    # -----------------
    # Other categorical -> keep multi-level rows (header + indented levels)
    # (You didn't ask to change this; only binary needed collapsing.)
    # -----------------
    ctab = pd.crosstab(dat[TARGET], dat[var], dropna=False)
    pv = pvalue_categorical(ctab)

    rows.append([var, "", "", fmt_p(pv)])
    for lv in list(ctab.columns):
        k0 = int(ctab.loc[0, lv]) if 0 in ctab.index else 0
        k1 = int(ctab.loc[1, lv]) if 1 in ctab.index else 0
        rows.append([f"  {lv}", fmt_n_pct(k0, n0), fmt_n_pct(k1, n1), ""])

table1 = pd.DataFrame(
    rows,
    columns=[
        "Variable",
        f"Readmission=0 (n={n0})",
        f"Readmission=1 (n={n1})",
        "P value",
    ],
)

# -----------------------
# 3) Save FULL table (ALL features)
# -----------------------
docx_full = out_dir / "Table1_descriptive_ALL_features.docx"
csv_full  = out_dir / "Table1_descriptive_ALL_features.csv"

build_table1_docx(
    table1,
    docx_full,
    title="Table 1. Baseline characteristics by readmission status (all variables; continuous = mean ± sd)"
)
table1.to_csv(csv_full, index=False)

print("Saved FULL Table 1:", docx_full)
print("Saved FULL Table 1:", csv_full)
print("Rows (full):", len(table1))


In [ ]:
# =======================
# TABLE 2 (FINAL — force RF to match your optimal setup)
# Uses:
# - data: cleaned_dataset_row2_imputed.csv (you already loaded into df earlier)
# - TARGET, BEST_SEED, BEST_TEST_SIZE
# - BEST_FEATURES (fixed list you provided)
# - FIXED_RF_PARAMS (fixed dict you provided)
#
# Produces:
# - Table2_train_leaky.csv
# - Table2_test.csv
# - Table2_train_leaky.docx
# - Table2_test.docx
# =======================

import time
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, recall_score, accuracy_score
)
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from sklearn.model_selection import ParameterGrid

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.shared import OxmlElement, qn


# -----------------------
# 0) User-fixed optimal setup (DO NOT CHANGE)
# -----------------------
TARGET = "Readmission"
BEST_SEED = 103
BEST_TEST_SIZE = 0.20

BEST_FEATURES = [
    "Albumin", "Alcohol Use", "Hypertension", "Immunocompromised", "CVD",
    "Num Dis Vessels", "Prior MI", "Hematocrit", "PAD", "Sleep Apnea",
    "Creatinine", "Gender", "Pneumonia", "tobacco_use"
]

FIXED_RF_PARAMS = dict(
    bootstrap=True,
    class_weight="balanced_subsample",
    max_depth=None,
    max_features="sqrt",
    min_impurity_decrease=1e-4,
    min_samples_leaf=20,
    min_samples_split=2,
    n_estimators=1000,
    n_jobs=-1,
)

# -----------------------
# 1) Sanity checks (expects you already have df + out_dir)
# -----------------------
assert "df" in globals(), "df not found. Please load cleaned_dataset_row2_imputed.csv into df first."
assert "out_dir" in globals(), "out_dir not found. Please define out_dir first."
out_dir = Path(out_dir)

need_cols = [TARGET] + BEST_FEATURES
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in df: {missing}")

# -----------------------
# 2) Split (fixed) + build X/y
# -----------------------
X_all = df[BEST_FEATURES].copy()
y_all = df[TARGET].astype(int).copy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all,
    test_size=BEST_TEST_SIZE,
    random_state=BEST_SEED,
    stratify=y_all
)

print("Train n:", len(X_tr), "Test n:", len(X_te))
print("Train event rate:", float(np.mean(y_tr)), "Test event rate:", float(np.mean(y_te)))
print("Features:", list(X_tr.columns))

# -----------------------
# 3) Metrics helper
# -----------------------
def eval_metrics(y_true, proba, thr=0.5):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba).astype(float)
    pred = (proba >= thr).astype(int)
    return {
        "AUC": float(roc_auc_score(y_true, proba)),
        "PRAUC": float(average_precision_score(y_true, proba)),
        "F1": float(f1_score(y_true, pred, zero_division=0)),
        "Recall": float(recall_score(y_true, pred, zero_division=0)),
        "Accuracy": float(accuracy_score(y_true, pred)),
    }

# -----------------------
# 4) RF EXACT optimal (seed + FIXED_RF_PARAMS)
# -----------------------
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_for_cv = RandomForestClassifier(
    random_state=BEST_SEED,
    **FIXED_RF_PARAMS
)

cv_aucs = cross_val_score(
    rf_for_cv,
    X_tr, y_tr,
    scoring="roc_auc",
    cv=cv5,
    n_jobs=-1
)

print("\nRF Train 5CV AUROC folds:", np.round(cv_aucs, 4))
print("RF Train 5CV AUROC mean±sd:", f"{cv_aucs.mean():.4f} ± {cv_aucs.std(ddof=1):.4f}")

best_rf = clone(rf_for_cv)
best_rf.fit(X_tr, y_tr)

p_tr_rf = best_rf.predict_proba(X_tr)[:, 1]
p_te_rf = best_rf.predict_proba(X_te)[:, 1]

rf_train_leaky = eval_metrics(y_tr, p_tr_rf, thr=0.5)
rf_test = eval_metrics(y_te, p_te_rf, thr=0.5)

print("RF Train AUC (fit+eval on same train):", round(rf_train_leaky["AUC"], 4))
print("RF Test AUROC:", round(rf_test["AUC"], 4))

# -----------------------
# 5) XGB for Table 2 (fast tune by train-CV AUROC)
# -----------------------
pos = int(np.sum(np.asarray(y_tr) == 1))
neg = int(np.sum(np.asarray(y_tr) == 0))
scale_pos_weight = (neg / max(pos, 1))

xgb_grid = {
    "n_estimators": [300, 600, 900],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.06],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "reg_lambda": [1.0, 5.0],
}

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_xgb_params = None
best_xgb_cv = -np.inf

t0 = time.time()
for params in ParameterGrid(xgb_grid):
    m = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        n_jobs=-1,
        random_state=BEST_SEED,
        scale_pos_weight=scale_pos_weight,
        **params
    )
    scores = cross_val_score(m, X_tr, y_tr, scoring="roc_auc", cv=CV, n_jobs=-1)
    mean_auc = float(np.mean(scores))
    if mean_auc > best_xgb_cv:
        best_xgb_cv = mean_auc
        best_xgb_params = params

print("\nBest XGB train-CV AUROC:", round(best_xgb_cv, 4))
print("Best XGB params:", best_xgb_params)
print("XGB tuning time(s):", round(time.time() - t0, 1))

best_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=BEST_SEED,
    scale_pos_weight=scale_pos_weight,
    **best_xgb_params
)
best_xgb.fit(X_tr, y_tr)

# -----------------------
# 6) Baseline models (train now)
# -----------------------
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=3000,
        class_weight="balanced",
        random_state=BEST_SEED
    ))
])

svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=BEST_SEED
    ))
])

mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=400,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=BEST_SEED
    ))
])

print("\nTraining Logistic regression / SVM / Neural Network (MLP)...")
t1 = time.time()
logreg.fit(X_tr, y_tr)
svm.fit(X_tr, y_tr)
mlp.fit(X_tr, y_tr)
print("Done. time(s):", round(time.time() - t1, 1))

# -----------------------
# 7) Build Table 2 (two parts: train-leaky and test)
# -----------------------
models = {
    "RF": best_rf,
    "XGB": best_xgb,
    "SVM": svm,
    "Neural Network": mlp,
    "Logistic regression": logreg,
}

def make_table2(models: dict, X_tr, y_tr, X_te, y_te, thr=0.5):
    rows_train, rows_test = [], []

    for name, model in models.items():
        # train-leaky
        p_tr = model.predict_proba(X_tr)[:, 1]
        mt = eval_metrics(y_tr, p_tr, thr=thr)
        rows_train.append({"Model": name, **mt})

        # test
        p_te = model.predict_proba(X_te)[:, 1]
        me = eval_metrics(y_te, p_te, thr=thr)
        rows_test.append({"Model": name, **me})

    df_train = pd.DataFrame(rows_train)
    df_test  = pd.DataFrame(rows_test)

    cols = ["Model", "AUC", "PRAUC", "F1", "Recall", "Accuracy"]
    df_train = df_train[cols].reset_index(drop=True)
    df_test  = df_test[cols].reset_index(drop=True)

    return df_train, df_test

t2_train, t2_test = make_table2(models, X_tr, y_tr, X_te, y_te, thr=0.5)

# -----------------------
# 8) Save CSV
# -----------------------
t2_train_csv = out_dir / "Table2_train_leaky.csv"
t2_test_csv  = out_dir / "Table2_test.csv"
t2_train.to_csv(t2_train_csv, index=False)
t2_test.to_csv(t2_test_csv, index=False)
print("\nSaved:", t2_train_csv)
print("Saved:", t2_test_csv)

# -----------------------
# 9) Save DOCX (Word) for Table 2
# -----------------------
def set_cell_margins(cell, top=80, start=80, bottom=80, end=80):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcMar = OxmlElement("w:tcMar")
    for m, v in (("top", top), ("start", start), ("bottom", bottom), ("end", end)):
        node = OxmlElement(f"w:{m}")
        node.set(qn("w:w"), str(v))
        node.set(qn("w:type"), "dxa")
        tcMar.append(node)
    tcPr.append(tcMar)

def df_to_docx_table(df_: pd.DataFrame, out_path: Path, title: str):
    doc = Document()
    p = doc.add_paragraph(title)
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT

    table = doc.add_table(rows=1, cols=df_.shape[1])
    table.style = "Table Grid"

    # header
    hdr_cells = table.rows[0].cells
    for j, col in enumerate(df_.columns):
        hdr_cells[j].text = str(col)
        set_cell_margins(hdr_cells[j])

    # body
    for i in range(df_.shape[0]):
        row_cells = table.add_row().cells
        for j, col in enumerate(df_.columns):
            val = df_.iloc[i, j]
            if isinstance(val, float):
                txt = f"{val:.4f}"
            else:
                txt = "" if pd.isna(val) else str(val)
            row_cells[j].text = txt
            set_cell_margins(row_cells[j])
            for par in row_cells[j].paragraphs:
                par.alignment = WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.RIGHT

    style = doc.styles["Normal"]
    style.font.name = "Times New Roman"
    style.font.size = Pt(10)

    doc.save(out_path)

t2_train_docx = out_dir / "Table2_train_leaky.docx"
t2_test_docx  = out_dir / "Table2_test.docx"

df_to_docx_table(
    t2_train,
    t2_train_docx,
    title="Table 2 (train-leaky). Model performance (trained on train; evaluated on same train)"
)
df_to_docx_table(
    t2_test,
    t2_test_docx,
    title="Table 2 (test). Model performance (trained on train; evaluated on held-out test)"
)

print("Saved:", t2_train_docx)
print("Saved:", t2_test_docx)

# -----------------------
# 10) Display
# -----------------------
print("\nTable 2 (train-leaky):")
display(t2_train.style.format({c: "{:.4f}" for c in ["AUC","PRAUC","F1","Recall","Accuracy"]}))

print("\nTable 2 (test):")
display(t2_test.style.format({c: "{:.4f}" for c in ["AUC","PRAUC","F1","Recall","Accuracy"]}))
